In [ ]:
# nicole opening the parks data from the NYC

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt

In [ ]:
parks_data = gpd.read_file("../Data/Parks_Properties_20260416.csv")

In [ ]:
parks_data

In [ ]:
# Set geometry from the WKT multipolygon column
parks_data = parks_data.set_geometry(
    gpd.GeoSeries.from_wkt(parks_data["multipolygon"]), crs="EPSG:4326"
)
parks_data = parks_data.to_crs("EPSG:32618")

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
parks_data.plot(ax=ax, edgecolor="black", facecolor="steelblue", alpha=0.5)
ax.set_title("NYC Parks Properties")
plt.tight_layout()
plt.show()

In [ ]:
NY_block_all = gpd.read_file(
    "../Data/drive-download-20260415T180155Z-3-001/nhgis0003_shape/nhgis0003_shape/nhgis0003_shapefile_tl2020_360_block_2020/NY_block_2020.shp"
)
NYC_FIPS = ["005", "047", "061", "081", "085"]
BOROUGH_NAMES = {
    "005": "Bronx",
    "047": "Brooklyn",
    "061": "Manhattan",
    "081": "Queens",
    "085": "Staten Island",
}
nyc_blocks = NY_block_all[NY_block_all["COUNTYFP20"].isin(NYC_FIPS)].copy()
nyc_blocks["borough"] = nyc_blocks["COUNTYFP20"].map(BOROUGH_NAMES)

# Dissolve blocks to create borough-level geometries
nyc_boundary = nyc_blocks.dissolve().to_crs("EPSG:32618")
nyc_boundary.to_file("nyc_boundary.geojson", driver="GeoJSON")

In [ ]:
# --- Build raster grid ---
resolution = 100  # meters per pixel
minx, miny, maxx, maxy = nyc_boundary.total_bounds
width = int((maxx - minx) / resolution)
height = int((maxy - miny) / resolution)
transform = from_bounds(minx, miny, maxx, maxy, width, height)

# --- Rasterize parks & boundary ---
park_mask = rasterize(
    shapes=parks_data.geometry,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    default_value=1,
    dtype="uint8",
)

nyc_mask = rasterize(
    shapes=nyc_boundary.geometry,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    default_value=1,
    dtype="uint8",
)

In [ ]:
# --- Distance transform ---
parks_distance_meters = distance_transform_edt(1 - park_mask) * resolution
parks_distance_meters[nyc_mask == 0] = np.nan

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(parks_distance_meters, cmap="RdYlGn_r")
plt.colorbar(im, ax=ax, label="Distance to nearest park (m)")
ax.set_title("Distance to Parks - NYC")
plt.tight_layout()
plt.show()

In [ ]:
with rasterio.open(
    "../Data/distance_to_parks.tif",
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:32618",
    transform=transform,
    nodata=np.nan,
) as dst:
    dst.write(parks_distance_meters.astype("float32"), 1)

In [ ]:
from rasterstats import zonal_stats

# Make sure blocks are in the same CRS as the raster
nyc_blocks_utm = nyc_blocks.to_crs("EPSG:32618")

# Compute mean distance to park for each block
stats = zonal_stats(
    nyc_blocks_utm,
    "../Data/distance_to_parks.tif",
    stats=["mean", "min", "max"],
    nodata=np.nan,
)

In [ ]:
nyc_blocks_utm["mean_dist_to_park"] = [s["mean"] for s in stats]
nyc_blocks_utm["min_dist_to_park"] = [s["min"] for s in stats]

In [ ]:
nyc_blocks_utm

In [ ]:
NYC_waterbodies = gpd.read_file("../Data/NYC_Hydrography.csv")

In [ ]:
NYC_waterbodies

In [ ]:
# Set geometry from the WKT multipolygon column
NYC_waterbodies = NYC_waterbodies.set_geometry(
    gpd.GeoSeries.from_wkt(NYC_waterbodies["the_geom"]), crs="EPSG:4326"
)
NYC_waterbodies = NYC_waterbodies.to_crs("EPSG:32618")

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
NYC_waterbodies.plot(ax=ax, edgecolor="black", facecolor="steelblue", alpha=0.5)
ax.set_title("NYC Waterbodies")
plt.tight_layout()
plt.show()

In [ ]:
# Use the same grid as before
water_mask = rasterize(
    shapes=NYC_waterbodies.geometry,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    default_value=1,
    dtype="uint8",
)

# --- Distance transform BEFORE applying NYC mask ---
distance_water_meters = distance_transform_edt(1 - water_mask) * resolution

# --- THEN mask to NYC ---
distance_water_meters[nyc_mask == 0] = np.nan

In [ ]:
with rasterio.open(
    "../Data/distance_to_water.tif",
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:32618",
    transform=transform,
    nodata=np.nan,
) as dst:
    dst.write(distance_water_meters.astype("float32"), 1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(distance_water_meters, cmap="RdYlGn_r")
plt.colorbar(im, ax=ax, label="Distance to nearest water Body (m)")
ax.set_title("Distance to Water Bodies - NYC")
plt.tight_layout()
plt.show()

In [ ]:
water_stats = zonal_stats(
    nyc_blocks_utm,
    "../Data/distance_to_water.tif",
    stats=["mean", "min", "max"],
    nodata=np.nan,
)

nyc_blocks_utm["mean_dist_to_water"] = [sw["mean"] for sw in water_stats]
nyc_blocks_utm["min_dist_to_water"] = [sw["min"] for sw in water_stats]

In [ ]:
nyc_blocks_utm

In [ ]:
nyc_boroughs_utm = nyc_blocks.dissolve(by="borough").to_crs("EPSG:32618")


fig, axes = plt.subplots(1, 2, figsize=(20, 8))

nyc_blocks_utm.plot(
    column="min_dist_to_park",
    ax=axes[0],
    cmap="RdYlGn_r",
    legend=True,
    legend_kwds={"label": "Min Distance to Park (m)"},
)
nyc_boroughs_utm.boundary.plot(ax=axes[0], edgecolor="black", linewidth=1.5)
axes[0].set_title("Min Distance to Parks by Census Block")
axes[0].set_axis_off()

nyc_blocks_utm.plot(
    column="min_dist_to_water",
    ax=axes[1],
    cmap="Blues_r",
    legend=True,
    legend_kwds={"label": "Min Distance to Water (m)"},
)
nyc_boroughs_utm.boundary.plot(ax=axes[1], edgecolor="black", linewidth=1.5)
axes[1].set_title("Min Distance to Water by Census Block")
axes[1].set_axis_off()

plt.suptitle(
    "NYC Access to Green Space and Water", fontsize=16, fontweight="bold", y=1.05
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(water_mask, cmap="Blues")
plt.title("Water Mask")
plt.show()

In [ ]:
nyc_blocks_utm.to_csv("../Data/NYC_block_temp_water.csv", sep=",")